# Lecture 7 — Sentiment analysis with a model on your own machine

**Module 3: Pythonic LLM Engineering**

This morning in Module 2 you met **classifiers**. This afternoon you meet one at work: on real sentences, running
on your own laptop, and measured properly rather than admired.

---

### What you need

Either of these works:

- **Your own project**, with the installation guide finished. Save this file *inside* your project folder, next
  to `pyproject.toml`. Then open the project **folder** in VS Code and pick the `.venv` kernel, or run
  `uv run jupyter lab` from that folder. Either is fine; what matters is that it runs inside the project.
- **Google Colab**, if the installation did not work: go to colab.research.google.com, choose
  *File → Upload notebook*, and upload this file. The first cell installs whatever Colab is missing.

No API key today. Nothing you type leaves the machine this notebook is running on.

### How this notebook is organised

Twelve code cells. **They all already work** — run them, read the explanation above each one, and write your
answers in the *Questions* cells. You only write code in the very last section.

| Part | What it does | In class |
|---|---|---|
| 0 | Check your setup and find your hardware | **Lab A**, 18 min |
| 1 | Your first classifier, and the sentences that fool it | " |
| 2 | What the model actually is: tokenizer, logits, softmax | **Lab B**, 12 min |
| 3 | Where it runs: CPU, GPU, or Apple silicon | **Lab C**, 18 min |
| 4 | Measure it: accuracy, confusion matrix, F1, AUC | " |
| 5 | What this model cannot do | " |
| Your turn | Three small tasks | at home |

---
## First: what is sentiment analysis?

Sentiment analysis means taking something a person wrote and putting a label on **how they felt** about what they
were writing about.

```
"The nurse explained everything clearly."        →   POSITIVE   0.9998
"I waited two hours and nobody told me why."     →   NEGATIVE   0.9991
```

So it is a **classifier** — exactly the thing you studied this morning. One input, one label out of a fixed set.
The only new part is the input: a sentence somebody wrote, instead of a row of numbers.

It comes in three shapes:

| shape | labels | where you meet it |
|---|---|---|
| binary | positive / negative | today, and most tutorials |
| three-class | positive / neutral / negative | most real systems. `neutral` is the hardest class, because most sentences are neither |
| aspect-based | one score per thing mentioned | *"the food was great but the service was slow"* → food +, service − |

### Where this is actually used

The pattern is always the same: **more text than anyone can read**, and a decision that only needs a rough sort.

| who | the text they drown in | what the label is for |
|---|---|---|
| a web shop, an app store | thousands of reviews a week | catch the one-star spike after a release |
| brand and social media teams | posts, mentions, comments | see a complaint wave the hour it starts |
| customer support | incoming email and chat | put the angry ones in front of a human first |
| a hospital or GP practice | free text on patient-experience surveys | thousands of comments a quarter nobody reads |
| HR | employee and exit surveys | which team's morale moved, and when |
| finance | headlines, filings, earnings calls | a signal, alongside the numbers |
| research | open answers in a questionnaire | coding 5,000 answers without an army of students |

Notice what is *not* on that list: deciding what happens to one particular person's message.

### Three ways to build that classifier

| approach | how it decides | what it costs you | where it breaks |
|---|---|---|---|
| a word list (VADER, Pattern) | somebody scored every English word by hand; add them up | minutes; no training, no data | "not bad", irony, and any word your field uses differently |
| classical ML (bag of words + naive Bayes / logistic regression) | learns from labelled examples which words predict which label | an afternoon, plus labelled data | word order, and words it never saw in training |
| **a transformer** (today) | reads the whole sentence in context, then classifies it | a 270 MB download | irony, and text from a field it was not trained on |

Rows two and three are both classifiers in this morning's sense. What changed is **where the features come
from**: in row two you chose them (words, counts, n-grams); in row three the model learned them, and somebody
else paid for that training. You will see both halves in Part 2.

> **One warning before we start.** Sentiment is not *topic*. The router you built in Lecture 6 sorted messages by
> what they were about. This sorts them by how the writer felt. Part 5 is what happens when you confuse the two.

---
# Part 0 — Your setup

Two libraries do the work today:

- **`transformers`** is Hugging Face's library. It loads any of the hundreds of thousands of models published on
  the Hugging Face **Hub**, all the same way.
- **`torch`** (PyTorch) is the engine underneath: it does the actual arithmetic. You do not have to learn PyTorch
  today.

The cell below does three things: it checks that both libraries are actually there, it prints their versions,
and it works out what hardware you have. If something is missing it tells you the exact command to run — and in
Colab it installs what is needed by itself.

Those three lines about the device matter more than they look. Plenty of tutorials write `device=0`, which means
*"the first NVIDIA graphics card"* — and crashes on every Mac and most laptops. Asking the machine first is three
lines, and then the same notebook runs everywhere.

In [3]:
import importlib.util
import subprocess
import sys
import time
from collections import Counter

# Is everything installed? In Colab, install it; anywhere else, say exactly what to run.
IN_COLAB = "google.colab" in sys.modules
missing = [m for m in ("transformers", "torch") if importlib.util.find_spec(m) is None]

if missing and IN_COLAB:
    print("Colab: installing", ", ".join(missing), "...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif missing:
    print("-" * 72)
    print("Not installed here:", ", ".join(missing))
    print()
    print("Close this notebook. In the folder that holds pyproject.toml, run:")
    print("    uv add 'transformers[torch]' jupyterlab")
    print("    uv run jupyter lab")
    print()
    print("then open this file again. See the installation guide if that fails.")
    print("-" * 72)
    raise RuntimeError("the libraries above are not installed - see the message")

import torch
import transformers

# Which hardware can PyTorch use on this machine?
#   "cuda" = an NVIDIA graphics card   (gaming laptops, servers, Google Colab)
#   "mps"  = Apple silicon             (M1/M2/M3/M4 Macs)
#   "cpu"  = the processor             (everybody else, and it is fine)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("transformers :", transformers.__version__)
print("torch        :", torch.__version__)
print("device       :", DEVICE)

transformers : 5.17.0
torch        : 2.14.0
device       : mps


`cpu` is a perfectly good answer: today's model is small and everything here runs on a processor in seconds.

If the cell told you something was missing, the usual cause is that the notebook is not running inside your
project. In VS Code: click the kernel name at the top right and choose the `.venv` in your project folder. In the
browser: close Jupyter and start it from the folder containing `pyproject.toml` with `uv run jupyter lab`. If that
still fails, use Colab for today and we will fix the installation afterwards.

---
# Part 1 — Your first classifier

## 1.1 One line, one model

`pipeline("sentiment-analysis")` does a surprising amount in one line:

1. it picks a model for this task (you did not name one, so it takes the default),
2. it **downloads** that model the first time — about 270 MB — into a cache folder on your machine,
3. it loads it into memory, ready to use.

The download happens once. After that the model works with the wifi switched off.

Watch the shape of what comes back: it is a **list**, with one dictionary per sentence, even when you gave it a
single sentence. Forgetting the `[0]` is the most common error of the afternoon.

In [4]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", device=DEVICE)

result = classifier("The nurse explained everything clearly.")

print(result)                          # a LIST, with one dict inside
print("label :", result[0]["label"])
print("score :", result[0]["score"])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9964839220046997}]
label : POSITIVE
score : 0.9964839220046997


## 1.2 Five sentences that decide today's lesson

Before you run this cell: **write down your own prediction for each of the five**, POSITIVE or NEGATIVE.

Four of them come from the online course this lab is based on. The fifth is ours, and it is the one that matters
in a clinic: in a hospital report a *negative* result is the best news of the week.

This cell also shows the other way to call a pipeline — give it a **list**, get a list back. That is not just
convenience; it is much faster, and Part 3 measures how much.

In [6]:
tricky = [
    "This show was not interesting",
    "This show was interesting",
    "This show was not bad at all",
    "I can't say that this was a good movie",
    "The biopsy came back negative and I am so relieved",
]

for text, r in zip(tricky, classifier(tricky)):        # a list in, a list out
    print(f"{r['label']:<9} {r['score']:.3f}   {text}")

NEGATIVE  1.000   This show was not interesting
POSITIVE  1.000   This show was interesting
POSITIVE  0.999   This show was not bad at all
NEGATIVE  0.928   I can't say that this was a good movie
POSITIVE  0.847   The biopsy came back negative and I am so relieved


### Questions — Part 1

1. Which of the five did your group predict wrong, and which did the model get wrong?
2. Sentence 5 is about a biopsy. In one sentence: why does a model trained on film reviews get it wrong?
3. Every sentence got a confident score, including the ones it got wrong. What does 0.99 actually tell you, and
   what does it not tell you?

---

*Answers:*

> 1. We have guessed all correctly.
>
> 2. I guess it could get it wrong because it is using 'negative' in a positive way, or maybe because the word 'biopsy' was not in the corpus. However it predicted the right tag, albeit it with the lowest score, so not sure if this notebook is broken again? Sorry that was more than one sentence.
>
> 3. It tells you the chance the predicted tag is correct. It does not tell you which tag it predicted.

---
# Part 2 — What the model actually is

## 2.1 Its name, and how it reads

A pipeline is three things bundled together: a **tokenizer**, a **model**, and a bit of code that turns numbers
back into words. This cell opens the first two.

The model's name is a specification. `distilbert-base-uncased-finetuned-sst-2-english` promises: a distilled
BERT (small), the middle size of that family, lower-cased (so *Clinical* and *clinical* are the same word),
fine-tuned on SST-2 (film reviews, positive or negative), English only.

The tokenizer chops text into pieces it knows. Look for `##` in the output — it marks a piece that was glued onto
the one before, because the whole word was not in the vocabulary. Watch what happens to *salbutamol*.

In [7]:
print("model      :", classifier.model.name_or_path)
print("its labels :", classifier.model.config.id2label)
print("size       :", f"{classifier.model.num_parameters():,} parameters")

tokenizer = classifier.tokenizer
text = "The nurse explained my salbutamol inhaler clearly."

ids = tokenizer(text)["input_ids"]
print()
print("tokens     :", tokenizer.convert_ids_to_tokens(ids))
print("vocabulary :", tokenizer.vocab_size, "word pieces")
print("max length :", tokenizer.model_max_length, "tokens")

model      : distilbert/distilbert-base-uncased-finetuned-sst-2-english
its labels : {0: 'NEGATIVE', 1: 'POSITIVE'}
size       : 66,955,010 parameters

tokens     : ['[CLS]', 'the', 'nurse', 'explained', 'my', 'sal', '##bu', '##tam', '##ol', 'in', '##hale', '##r', 'clearly', '.', '[SEP]']
vocabulary : 30522 word pieces
max length : 512 tokens


## 2.2 The two numbers the model really produces

Now the same prediction by hand, so you can see the machinery.

The model turns the token IDs into **two numbers** — one per label. They are called *logits*: raw scores, not
probabilities, and they can be negative. `softmax` turns any list of scores into probabilities that add up to 1.

This is the sentence from Lecture 1 with the lid off. There, a model scored two hundred thousand possible tokens
and you could see none of them. Here there are exactly two, and you can print them both.

In [16]:
model = classifier.model
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():                  # predicting, not training
    logits = model(**inputs).logits[0]

probs = logits.softmax(-1)

print("labels :", model.config.id2label)
print("logits :", [round(v, 3) for v in logits.tolist()])
print("probs  :", [round(v, 3) for v in probs.tolist()])

winner = int(probs.argmax())
print()
print(f"so: {model.config.id2label[winner]}  {probs[winner]:.3f}")
print("and the pipeline said:", classifier(text)[0])

labels : {0: 'NEGATIVE', 1: 'POSITIVE'}
logits : [-2.148, 2.056]
probs  : [0.015, 0.985]

so: POSITIVE  0.985
and the pipeline said: {'label': 'POSITIVE', 'score': 0.9852880835533142}


**This is the connection to this morning.** Those two numbers come out of a single linear layer sitting on top of
the transformer — a classifier of exactly the kind you studied, with two outputs. Everything underneath it is
feature extraction: 66 million parameters whose whole job is to turn a sentence into a vector that a simple
classifier can separate.

That is the real difference from bag-of-words. Not the classifier at the end — that part is ordinary. The
features, which were learned from millions of sentences instead of chosen by you.

And notice what is missing from the picture: no prompt, no system message, no instructions. To change what this
model does you cannot talk to it. You have to change the model.

### Questions — Part 2

1. How many tokens did the sentence become, and how many pieces did `salbutamol` break into? Why that word?
2. Write down your two logits and the two probabilities. Which is bigger, and by how much?
3. This model has 66 million parameters and a 512-token limit; the Groq model has about 20 billion and over
   100,000. Name one task from earlier lectures that only the big one can do.

---

*Answers:*

> 1. 15, including punctuation, the beginning, and the ending token. Salbutamol changes into 4 pieces. It did that because it does not exist in the vocabulary of the tokenizer.
>
> 2. logits : [-2.148, 2.056] probs  : [0.015, 0.985]. The logits has a smaller negative (negative) number and a bigger positive (positive) number. The probabilities are between 0 and 1, so smaller for the positive prediction and bigger for the negative prediction.
>
> 3. Writing actual texts, like describing the symptoms for certain ailments.

---
# Part 3 — Where does it run, and does it matter?

One cell, two questions. First: what hardware did PyTorch find? Second: does calling the model **forty times**
cost the same as calling it **once with forty sentences**?

It does not, and the reason is worth knowing. A GPU does not make one sentence much faster — there is a fixed
cost to moving data onto the card, and this model is tiny. What a GPU does is thousands of multiplications at the
same time, which only helps if you hand it thousands at once. So the interesting comparison is not CPU against
GPU; it is one-at-a-time against batched, and that one applies on every machine in the room.

> **If you own an NVIDIA laptop and still see `cpu`, nothing is broken.** The PyTorch package we installed is the
> CPU build on Windows and Mac; only on Linux does the default one carry CUDA. That is deliberate: a CUDA build is
> a much larger download and this model does not need it. Nothing in this notebook runs slowly on a processor.

In [9]:
print("CUDA (NVIDIA) :", torch.cuda.is_available())
print("MPS (Apple)   :", torch.backends.mps.is_available())
print("using         :", DEVICE)

sample = tricky * 8                    # 40 sentences

t0 = time.perf_counter()
for s in sample:                       # one call per sentence
    classifier(s)
one_at_a_time = time.perf_counter() - t0

t0 = time.perf_counter()
classifier(sample, batch_size=8)       # one call for all of them
batched = time.perf_counter() - t0

print()
print(f"one at a time : {one_at_a_time:.2f} s   ({one_at_a_time / len(sample) * 1000:.0f} ms per sentence)")
print(f"batched       : {batched:.2f} s   ({batched / len(sample) * 1000:.0f} ms per sentence)")
print(f"speed-up      : {one_at_a_time / batched:.1f}x on {DEVICE}")

CUDA (NVIDIA) : False
MPS (Apple)   : True
using         : mps

one at a time : 0.18 s   (4 ms per sentence)
batched       : 0.34 s   (9 ms per sentence)
speed-up      : 0.5x on mps


### Questions — Part 3

1. What hardware did your machine report, and how many milliseconds did one sentence take?
2. How much faster was the batch? Compare with somebody whose device line says something different.
3. A Groq call in Lecture 6 took a few hundred milliseconds. Where does that time go, and why is it not the model
   being slow?

---

*Answers:*

> 1. MPS Apple, 4ms per sentence. 
>
> 2. The batch was twice slower. Nvidia GPUs had a speedup.
>
> 3. That time is mostly spent in transit during the API calls. The model itself is hosted on a powerful machine, with dedicated AI hardware.

---
# Part 4 — Measure it properly

Five interesting sentences told you the model has weak spots. They cannot tell you **how often** it is wrong, and
that is the only question an engineer can act on.

So: eighty labelled sentences of patient feedback, written for this course. Forty happy, forty unhappy, and
deliberately awkward — negations, irony, mixed sentences, four Dutch lines, and six where clinical vocabulary
inverts the everyday meaning.

Everything from here on is plain Python. No scikit-learn: you already know these formulas, and writing them out
is three minutes of revision that also removes an installation.

In [17]:
# (text, label) -- written for this course, so no licence problems and no downloads.
FEEDBACK = [
    # --- plain positive ---------------------------------------------------
    ("The nurse explained everything clearly and I felt calm.", "positive"),
    ("I was seen five minutes after I arrived.", "positive"),
    ("The doctor listened to me without interrupting.", "positive"),
    ("Booking the appointment online took less than a minute.", "positive"),
    ("The ward was clean and quiet, I slept well.", "positive"),
    ("Everyone was friendly from the reception desk onwards.", "positive"),
    ("My questions were answered in plain language.", "positive"),
    ("The physiotherapist gave me exercises that actually helped.", "positive"),
    ("I got my results the same day.", "positive"),
    ("The pharmacy had my prescription ready when I arrived.", "positive"),
    ("The surgeon phoned me personally the evening before.", "positive"),
    ("Parking was easy and the signs were clear.", "positive"),
    ("They found an interpreter for my mother within ten minutes.", "positive"),
    ("The follow-up call was a lovely surprise.", "positive"),
    ("I felt safe the whole time I was there.", "positive"),
    ("The new online portal is much better than the old one.", "positive"),
    ("My treatment plan was explained with a drawing I could keep.", "positive"),
    ("The night staff checked on me without waking me.", "positive"),
    ("I was treated with respect by every single person.", "positive"),
    ("The waiting room had space for my wheelchair.", "positive"),
    ("The anaesthetist talked me through every step beforehand.", "positive"),
    # --- plain negative ---------------------------------------------------
    ("I waited two hours and nobody told me why.", "negative"),
    ("The receptionist was rude when I asked a simple question.", "negative"),
    ("My appointment was cancelled twice without an explanation.", "negative"),
    ("Nobody answered the phone all morning.", "negative"),
    ("The room was dirty and the bin had not been emptied.", "negative"),
    ("I was given the wrong dose and only noticed it myself.", "negative"),
    ("The doctor spent the whole consultation looking at the screen.", "negative"),
    ("I still have not received the letter they promised three weeks ago.", "negative"),
    ("The parking charges are ridiculous for a ten-minute visit.", "negative"),
    ("They lost my referral and I had to start again.", "negative"),
    ("The instructions on the medication were impossible to understand.", "negative"),
    ("I was woken at six for a test that never happened.", "negative"),
    ("My pain was dismissed as stress without any examination.", "negative"),
    ("The online system logged me out four times.", "negative"),
    ("No one explained what the procedure involved.", "negative"),
    ("I had to repeat my history to five different people.", "negative"),
    ("The discharge was rushed and I left confused.", "negative"),
    ("They spoke about me as if I were not in the room.", "negative"),
    ("The waiting area was freezing cold.", "negative"),
    ("My complaint was never acknowledged.", "negative"),
    # --- negations --------------------------------------------------------
    ("The wait was not bad at all.", "positive"),
    ("I can't say that I was unhappy with the care.", "positive"),
    ("This was not the disaster I had feared.", "positive"),
    ("Nothing went wrong, which is more than I expected.", "positive"),
    ("I would not call the service excellent.", "negative"),
    ("It is not the worst hospital I have been in, but only just.", "negative"),
    ("I can't say that the doctor was helpful.", "negative"),
    ("The appointment was not useless, although it was close.", "negative"),
    # --- mixed, with one side clearly dominant ----------------------------
    ("The doctor was kind, but I waited three hours and left in tears.", "negative"),
    ("Parking was a nightmare, but the care I received was outstanding.", "positive"),
    ("Long wait, wonderful nurses, and I would come back.", "positive"),
    ("The food was fine; everything else was a mess.", "negative"),
    ("Friendly staff cannot make up for being sent to the wrong clinic twice.", "negative"),
    ("The building is old, but I have never been treated better.", "positive"),
    # --- clinical vocabulary: 'negative' and 'positive' flip meaning -------
    ("The biopsy came back negative and I am so relieved.", "positive"),
    ("My test was negative, thank goodness.", "positive"),
    ("The scan was clear, no sign of anything.", "positive"),
    ("They told me the result was positive and my world stopped.", "negative"),
    ("A positive test on my birthday, of all days.", "negative"),
    ("The infection markers are negative, so I can go home.", "positive"),
    # --- understatement and irony -----------------------------------------
    ("Three cancelled appointments in one month. Excellent work.", "negative"),
    ("Wonderful, another form to fill in.", "negative"),
    ("Only a four-hour wait this time, so that is progress.", "negative"),
    ("I suppose the coffee machine worked.", "negative"),
    # --- Dutch ------------------------------------------------------------
    ("De verpleegkundige was heel vriendelijk en nam de tijd voor mij.", "positive"),
    ("Ik heb drie uur gewacht en niemand kwam iets uitleggen.", "negative"),
    ("De afspraak werd zonder bericht geannuleerd.", "negative"),
    ("Ik werd snel geholpen en goed geïnformeerd.", "positive"),
    # --- short and blunt ---------------------------------------------------
    ("Excellent.", "positive"),
    ("Never again.", "negative"),
    ("Thank you for everything.", "positive"),
    ("Deeply disappointing.", "negative"),
    ("Could not fault them.", "positive"),
    ("Avoid this clinic.", "negative"),
    ("Everything on time, everything explained.", "positive"),
    ("A complete waste of an afternoon.", "negative"),
    ("I am grateful to the whole team.", "positive"),
    ("Worst experience of my life.", "negative"),
    ("Five stars, no notes.", "positive"),
]

texts = [t for t, _ in FEEDBACK]
gold = [g for _, g in FEEDBACK]

print(len(FEEDBACK), "sentences")
print(Counter(gold))

80 sentences
Counter({'positive': 40, 'negative': 40})


## 4.2 Predict, count, and lay the mistakes out in a square

`accuracy` is how many you got right over how many there were. It is the obvious number and it hides the
important one, because it treats every mistake as the same mistake.

The **confusion matrix** does not. Four counts, and every other number below is built from them. The naming trips
people up, so here is the trick: *the second word is what the model said, the first is whether it was right*. A
false positive is the model saying positive and being wrong.

In a complaints inbox those four boxes have very different costs. An unhappy patient filed as happy is a
complaint nobody ever reads again. A happy patient flagged as unhappy costs someone ten minutes.

In [18]:
outputs = classifier(texts, batch_size=8)

pred = ["positive" if o["label"] == "POSITIVE" else "negative" for o in outputs]

# the probability that the sentence is POSITIVE, whichever label won
prob_positive = [o["score"] if o["label"] == "POSITIVE" else 1 - o["score"] for o in outputs]

correct = sum(p == g for p, g in zip(pred, gold))
print(f"accuracy: {correct}/{len(gold)} = {correct / len(gold):.2f}")
print("(the model card advertises 0.91 - measured on film reviews, its own test set)")

TP = sum(p == "positive" and g == "positive" for p, g in zip(pred, gold))
FP = sum(p == "positive" and g == "negative" for p, g in zip(pred, gold))
FN = sum(p == "negative" and g == "positive" for p, g in zip(pred, gold))
TN = sum(p == "negative" and g == "negative" for p, g in zip(pred, gold))

print()
print("                  said positive    said negative")
print(f"really positive  {TP:>10}       {FN:>10}")
print(f"really negative  {FP:>10}       {TN:>10}")

accuracy: 56/80 = 0.70
(the model card advertises 0.91 - measured on film reviews, its own test set)

                  said positive    said negative
really positive          22               18
really negative           6               34


## 4.3 Precision, recall, F1 — and AUC

- **precision** = of the sentences it called happy, how many really were. *When it says happy, can I trust it?*
- **recall** = of the genuinely happy sentences, how many it found. *Is it missing them?*
- **F1** = one number that punishes you for being bad at either. It is the *harmonic* mean, not the ordinary
  average, for a reason: the ordinary average of 1.0 and 0.0 is a respectable-looking 0.5, while the harmonic
  mean is 0.0. It refuses to reward a model that has given up on one of the two.

All three depend on where you put the 0.5 boundary. **AUC** does not. It asks one question instead: take a random
happy sentence and a random unhappy one — how often does the happy one get the higher score? 1.0 means always,
0.5 means a coin flip. The cell computes it by doing literally that, over all 40 × 40 pairs.

In [19]:
def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


p_pos, r_pos, f1_pos = prf(TP, FP, FN)
p_neg, r_neg, f1_neg = prf(TN, FN, FP)     # the same four counts, from the other side

print(f"{'class':<10}{'precision':>11}{'recall':>9}{'F1':>8}")
print("-" * 38)
print(f"{'positive':<10}{p_pos:>11.2f}{r_pos:>9.2f}{f1_pos:>8.2f}")
print(f"{'negative':<10}{p_neg:>11.2f}{r_neg:>9.2f}{f1_neg:>8.2f}")
print(f"\nmacro F1 (the average of the two): {(f1_pos + f1_neg) / 2:.2f}")

# AUC: over every (happy, unhappy) pair, how often does the happy one score higher?
pos_scores = [s for s, g in zip(prob_positive, gold) if g == "positive"]
neg_scores = [s for s, g in zip(prob_positive, gold) if g == "negative"]

wins = sum((a > b) + 0.5 * (a == b) for a in pos_scores for b in neg_scores)
auc = wins / (len(pos_scores) * len(neg_scores))
print(f"AUC: {auc:.3f}   (over {len(pos_scores)} x {len(neg_scores)} = "
      f"{len(pos_scores) * len(neg_scores)} pairs)")

class       precision   recall      F1
--------------------------------------
positive         0.79     0.55    0.65
negative         0.65     0.85    0.74

macro F1 (the average of the two): 0.69
AUC: 0.829   (over 40 x 40 = 1600 pairs)


## 4.4 Look at the mistakes

This is the most useful cell in the notebook. Every number above collapses eighty decisions into one figure; this
prints the decisions it got wrong, and how sure it was about them.

Read them and sort them into groups: **negation**, **irony**, **mixed sentiment**, **clinical vocabulary**,
**Dutch**. Which group is biggest?

In [20]:
mistakes = [(t, g, p, s) for t, g, p, s in zip(texts, gold, pred, prob_positive) if p != g]

print(f"{len(mistakes)} mistakes out of {len(gold)}\n")
for t, g, p, s in mistakes:
    print(f"said {p:<9} (really {g:<9})  score_positive={s:.2f}")
    print(f"    {t}\n")

24 mistakes out of 80

said negative  (really positive )  score_positive=0.02
    I was seen five minutes after I arrived.

said negative  (really positive )  score_positive=0.00
    Booking the appointment online took less than a minute.

said negative  (really positive )  score_positive=0.00
    My questions were answered in plain language.

said negative  (really positive )  score_positive=0.43
    I got my results the same day.

said negative  (really positive )  score_positive=0.01
    The pharmacy had my prescription ready when I arrived.

said negative  (really positive )  score_positive=0.02
    The surgeon phoned me personally the evening before.

said negative  (really positive )  score_positive=0.02
    They found an interpreter for my mother within ten minutes.

said negative  (really positive )  score_positive=0.20
    My treatment plan was explained with a drawing I could keep.

said negative  (really positive )  score_positive=0.01
    The waiting room had space for my w

### Questions — Part 4

1. What accuracy did you get? How far is it from the advertised 0.91, and why is that gap not the model being
   broken?
2. Which mistake is more common in your confusion matrix: happy patients marked unhappy, or unhappy patients
   marked happy? Which would worry a hospital more?
3. Compare the F1 for `positive` with the F1 for `negative`. Is the model equally good at both?
4. Sort the mistakes in 4.4 into the five groups. Which is biggest? Which one would still defeat a model trained
   on our own data?
5. Your AUC is probably higher than your accuracy suggests. What does that tell you about where the 0.5 boundary
   sits?

---

*Answers:*

> 1. I got .7. The advertised .91 is based on the corpus that it was trained on. If the evaluated dataset does not match the domain or some other aspect of the initial corpus, the performance will drop.
>
> 2. Happy patients marked negative was the more often occurring mistake. This is the least worrying mistake since it does not mean complains are left on read.
>
> 3. The model has a higher F1 for predicting negative sentiment.
>
> 4. To be honest i find the list of "**negation**, **irony**, **mixed sentiment**, **clinical vocabulary**, **Dutch**" to be excluding a lot of the mistaken predictions. Besides the dutch texts, most predictions are just neutral and do not contain a lot of sentiment. 
>
> 5. On the higher end.


---
# Part 5 — What this model cannot do

Here are the twelve patient messages from Lecture 6, where you built a router that sorted them into
*appointment*, *prescription* and *clinical*.

Send them through a sentiment classifier and watch what happens. Every message gets a label, because the model
has exactly two and must pick one. Nothing is broken. The question was wrong.

In [21]:
TRIAGE = [
    ("Can I move my check-up to next Tuesday?", "appointment"),
    ("I need a repeat of my amlodipine 5 mg tablets.", "prescription"),
    ("My ankle has been swollen for 14 days now, should I worry?", "clinical"),
    ("Please cancel Thursday, I am away for work.", "appointment"),
    ("The pharmacy says my prescription has run out.", "prescription"),
    ("Is it normal to feel dizzy on this medication?", "clinical"),
    ("Do you have anything earlier than the 14th?", "appointment"),
    ("Could you send my inhaler prescription to a different chemist?", "prescription"),
    ("What do my cholesterol results mean?", "clinical"),
    ("I will not make my 9am slot tomorrow.", "appointment"),
    ("I have two tablets left and need more.", "prescription"),
    ("Mijn afspraak van morgen om 9 uur kan ik niet halen.", "appointment"),
]

for (msg, should_be), r in zip(TRIAGE, classifier([m for m, _ in TRIAGE])):
    print(f"{r['label']:<9} {r['score']:.2f}   should be: {should_be:<13} {msg[:50]}")

NEGATIVE  1.00   should be: appointment   Can I move my check-up to next Tuesday?
NEGATIVE  1.00   should be: prescription  I need a repeat of my amlodipine 5 mg tablets.
NEGATIVE  0.99   should be: clinical      My ankle has been swollen for 14 days now, should 
NEGATIVE  1.00   should be: appointment   Please cancel Thursday, I am away for work.
NEGATIVE  1.00   should be: prescription  The pharmacy says my prescription has run out.
NEGATIVE  1.00   should be: clinical      Is it normal to feel dizzy on this medication?
NEGATIVE  0.99   should be: appointment   Do you have anything earlier than the 14th?
NEGATIVE  1.00   should be: prescription  Could you send my inhaler prescription to a differ
NEGATIVE  1.00   should be: clinical      What do my cholesterol results mean?
NEGATIVE  1.00   should be: appointment   I will not make my 9am slot tomorrow.
NEGATIVE  1.00   should be: prescription  I have two tablets left and need more.
NEGATIVE  0.90   should be: appointment   Mijn afspra

### Questions — Part 5

1. The model called several of these POSITIVE. Is it wrong? Answer carefully.
2. You need a classifier for *appointment / prescription / clinical*. Four options: search the Hub for a closer
   model, zero-shot classification, fine-tune this one, or ask an LLM through an API. Which would you choose for
   a GP practice with 500 messages a day, and what would change your mind?

---

*Answers:*

> 1. No for me it called them all negative. I can see why, since they are medical problems and those might be present in certain ways to critique filmmakers. 
>
> 2. For a GP practice with 500 messages a day, I would not go for a self-hosted model. This only complicates things when it breaks. My recommendation would thus be to call an LLM that is externally hosted. If the amount of messages would be bigger, then the economics would shift towards using something self hosted. Then I would likely find a model on the hub. 

---
# Your turn

**Task 1 is required.** Write five sentences of your own that fool the model, each with the label you think is
right. Run the cell, then write one paragraph: what do your five have in common?

Two more, if you want them — both at home, not on the train:

- **Task 2 — a model that speaks Dutch.** Run `nlptown/bert-base-multilingual-uncased-sentiment` on the four
  Dutch lines in `FEEDBACK` and compare with today's model. What changed, and what did not?
  (`pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=DEVICE)`)
- **Task 3 — give the model our labels.** `pipeline("zero-shot-classification", device=DEVICE)` lets you supply
  `candidate_labels=["appointment", "prescription", "clinical"]` at run time. Try it on `TRIAGE` and score it.
  The model is 1.6 GB, so start the download before you make coffee.

In [35]:
# TASK 1 --------------------------------------------------------------
my_sentences = [
    # your five sentences here, each with the label you think is right
    # ("I have no complaints whatsoever.", "positive"),
    ("Ik vond dit een geweldige film.", "pos"),
    ("I don't like how good this movie was.", "pos"),
    ("Not a bad show, but all is said with that.", "neg"),
    ("I could care less about this movie.", "pos"),
    ("这是一部好电影", "pos"),
]

for text, my_label in my_sentences:
    r = classifier(text)[0]
    said = "positive" if r["label"] == "POSITIVE" else "negative"
    mark = "fooled it" if said != my_label else "got it right"
    print(f"{mark:<14} model said {said:<9} {r['score']:.2f}   {text}")

fooled it      model said negative  0.95   Ik vond dit een geweldige film.
fooled it      model said negative  0.99   I don't like how good this movie was.
fooled it      model said positive  1.00   Not a bad show, but all is said with that.
fooled it      model said negative  1.00   I could care less about this movie.
fooled it      model said negative  0.83   这是一部好电影


---
## Before you leave

- [x] Fill in the table below
- [x] Save this notebook and commit it next to your earlier work (Colab: *File → Download → .ipynb*)
- [x] Task 1 at home
- [x] Leave the model in its cache — `transformers` comes back later in the course

| measurement | value |
|---|---|
| your hardware (cpu / cuda / mps) | mps |
| ms per sentence: one at a time / batched | 4 ms / 9 ms |
| accuracy on the 80 sentences | 56/80 = 0.70 |
| F1 positive / F1 negative | 0.65 / 0.74 |
| AUC | 0.829 |
| the biggest group of mistakes | plain/neutral-looking positive patient feedback |
| one sentence: would you use this model in a clinic? | No. |